In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import math
import random
import seaborn as sns
import matplotlib.pyplot as plt
# Sirve para medir la velocidad de una función
import time
%load_ext line_profiler
import sys
sys.path.insert(1, '/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Codigos/Funciones_utiles')
import funciones_aux_bootstrap as fab
from matplotlib import legend_handler

# Fijamos el estilo de la gráfica
plt.style.use('petroff10')

# Votos dir vs Casillas

## No estratificado

In [76]:
# Leemos las bases
# Base de bootstrap por casillas sin estratificar
df_boostraps_c_s_e=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap por casillas/Distintas_muestras_casillas_bootstrap_sin_est_f_1.csv", index_col=0)
# Base de bootstrap por votos (directo)
df_boostraps_votos_se=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap votos/Directo/Distintas_muestras_bootstrap_votos_dir_sin_est_f_1.csv", index_col=0)

## Tabla candidatos con menor cobertura

In [93]:
# Para casillas
df_boostraps_c_m_c=df_boostraps_c_s_e[["Cobertura_m_candidato_nom", "Tamaño_muestra_casillas"]].copy()
df_boostraps_c_m_c["Tamaño_muestra_casillas"]=df_boostraps_c_m_c["Tamaño_muestra_casillas"].astype(int)
# Agrupamos por candidato
df_agg_cand_c=df_boostraps_c_m_c.groupby("Cobertura_m_candidato_nom").agg({'Tamaño_muestra_casillas':'unique'}).reset_index()
# Renombramos columnas
df_agg_cand_c=df_agg_cand_c.rename(columns={'Cobertura_m_candidato_nom':'Candidate with the lowest coverage', 'Tamaño_muestra_casillas':'Sample size'})
# Creamos una columna para identificar que se está muestreando
df_agg_cand_c["Sampling"]='Polling places'
df_agg_cand_c

,Candidate with the lowest coverage,Sample size,Sampling
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,"[100, 150, 200, 250, 300, 350, 400]",Polling places


In [94]:
# Para votos
df_boostraps_v_m_c=df_boostraps_votos_se[["Cobertura_m_candidato_nom", "Tamaño_muestra_votos"]].copy()
df_boostraps_v_m_c["Tamaño_muestra_votos"]=df_boostraps_v_m_c["Tamaño_muestra_votos"].astype(int)
df_agg_cand_v=df_boostraps_v_m_c.groupby("Cobertura_m_candidato_nom").agg({'Tamaño_muestra_votos':'unique'}).reset_index()
# Renombramos columnas
df_agg_cand_v=df_agg_cand_v.rename(columns={'Cobertura_m_candidato_nom':'Candidate with the lowest coverage', 'Tamaño_muestra_votos':'Sample size'})
# Creamos una columna para identificar que se está muestreando
df_agg_cand_v["Sampling"]='Votes'
df_agg_cand_v

,Candidate with the lowest coverage,Sample size,Sampling
0,RENAN_BARRERA_CONCHA,[5000],Votes
1,VIDA_ARAVARI_GOMEZ_HERRERA,[1000],Votes
2,VOTOS_NULOS_CAND_NO_REGIS,[10000],Votes
3,YAMIL_JASMIN_LOPEZ_MANRIQUE,"[500, 2000, 20000]",Votes


In [95]:
# Concatenamos las tablas
df_agg_com=pd.concat([df_agg_cand_c,df_agg_cand_v])
# Reordenamos las columnas
df_agg_com_f=df_agg_com[["Candidate with the lowest coverage","Sampling","Sample size"]].reset_index(drop=True)
df_agg_com_f

,Candidate with the lowest coverage,Sampling,Sample size
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places,"[100, 150, 200, 250, 300, 350, 400]"
1,RENAN_BARRERA_CONCHA,Votes,[5000]
2,VIDA_ARAVARI_GOMEZ_HERRERA,Votes,[1000]
3,VOTOS_NULOS_CAND_NO_REGIS,Votes,[10000]
4,YAMIL_JASMIN_LOPEZ_MANRIQUE,Votes,"[500, 2000, 20000]"


In [96]:
# Renombramos los nombres de los candidatos
df_agg_com_f["Candidate with the lowest coverage"]=["Jasmín López Manrique", "Renan Barrera Concha", "Vida Gómez Herrera", "Null votes and votes for unregistered candidates", "Jasmín López Manrique"]
df_agg_com_f

,Candidate with the lowest coverage,Sampling,Sample size
0,Jasmín López Manrique,Polling places,"[100, 150, 200, 250, 300, 350, 400]"
1,Renan Barrera Concha,Votes,[5000]
2,Vida Gómez Herrera,Votes,[1000]
3,Null votes and votes for unregistered candidates,Votes,[10000]
4,Jasmín López Manrique,Votes,"[500, 2000, 20000]"


In [97]:
print(df_agg_com_f.to_latex(index=False, label='tab:Lowest_cov_votos_vs_cas',caption='Candidates with the lowest coverage'))

\begin{table}
\caption{Candidates with the lowest coverage}
\label{tab:Lowest_cov_votos_vs_cas}
\begin{tabular}{lll}
\toprule
Candidate with the lowest coverage & Sampling & Sample size \\
\midrule
Jasmín López Manrique & Polling places & [100 150 200 250 300 350 400] \\
Renan Barrera Concha & Votes & [5000] \\
Vida Gómez Herrera & Votes & [1000] \\
Null votes and votes for unregistered candidates & Votes & [10000] \\
Jasmín López Manrique & Votes & [  500  2000 20000] \\
\bottomrule
\end{tabular}
\end{table}



## Diferencia en cobertura máxima y mínima por candidato

In [120]:
# Base para la casillas inicial
df_aux_c=df_boostraps_c_s_e[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff = df_aux_c.max() - df_aux_c.min()

# Lo pasamos a Data Frame
df_max_c=pd.DataFrame(max_diff).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (polling places)'})
df_max_c

,Candidates,Maximum difference in coverages (polling places)
0,Cob_JOAQUIN_DIAZ_MENA,0.028
1,Cob_RENAN_BARRERA_CONCHA,0.031
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.050
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.028
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.164


In [121]:
# Base para la votos inicial
df_aux_v=df_boostraps_votos_se[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff_v = df_aux_v.max() - df_aux_v.min()

# Lo pasamos a Data Frame
df_max_v=pd.DataFrame(max_diff_v).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (votes)'})
df_max_v

,Candidates,Maximum difference in coverages (votes)
0,Cob_JOAQUIN_DIAZ_MENA,0.014
1,Cob_RENAN_BARRERA_CONCHA,0.021
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.018
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.018
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.051


In [124]:
# Juntamos las bases
df_max_f=df_max_c.merge(df_max_v, how='inner', on='Candidates')
# Renombramos la columna de candidatos
df_max_f["Candidates"]=["Joaquín Díaz Mena", "Renan Barrera Concha", "Vida Gómez Herrera", "Null votes and votes for unregistered candidates","Jasmín López Manrique"]
df_max_f

,Candidates,Maximum difference in coverages (polling places),Maximum difference in coverages (votes)
0,Joaquín Díaz Mena,0.028,0.014
1,Renan Barrera Concha,0.031,0.021
2,Vida Gómez Herrera,0.050,0.018
3,Null votes and votes for unregistered candidates,0.028,0.018
4,Jasmín López Manrique,0.164,0.051


In [125]:
print(df_max_f.to_latex(index=False, label='tab:Max_diff_cov_votos_vs_cas',caption='Maximum difference in coverages'))

\begin{table}
\caption{Maximum difference in coverages}
\label{tab:Max_diff_cov_votos_vs_cas}
\begin{tabular}{lrr}
\toprule
Candidates & Maximum difference in coverages (polling places) & Maximum difference in coverages (votes) \\
\midrule
Joaquín Díaz Mena & 0.028000 & 0.014000 \\
Renan Barrera Concha & 0.031000 & 0.021000 \\
Vida Gómez Herrera & 0.050000 & 0.018000 \\
Null votes and votes for unregistered candidates & 0.028000 & 0.018000 \\
Jasmín López Manrique & 0.164000 & 0.051000 \\
\bottomrule
\end{tabular}
\end{table}



## Estratificado por distritos locales

In [3]:
# Leemos las bases
# Base de bootstrap por casillas sin estratificar
df_boostraps_c_est=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap por casillas/Distintas_muestras_casillas_bootstrap_estratificado_f_1.csv", index_col=0)

# Base de bootstrap por votos (directo)
df_boostraps_votos_est=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases finales rep/Bootstrap votos/Directo/Distintas_muestras_bootstrap_votos_dir_estratificado_f_1.csv", index_col=0)

## Tabla candidatos con menor cobertura

In [4]:
# Para casillas
df_boostraps_c_m_c_est=df_boostraps_c_est[["Cobertura_m_candidato_nom", "Tamaño_muestra_casillas"]].copy()
df_boostraps_c_m_c_est["Tamaño_muestra_casillas"]=df_boostraps_c_m_c_est["Tamaño_muestra_casillas"].astype(int)
# Agrupamos por candidato
df_agg_cand_c_est=df_boostraps_c_m_c_est.groupby("Cobertura_m_candidato_nom").agg({'Tamaño_muestra_casillas':'unique'}).reset_index()
# Renombramos columnas
df_agg_cand_c_est=df_agg_cand_c_est.rename(columns={'Cobertura_m_candidato_nom':'Candidate with the lowest coverage', 'Tamaño_muestra_casillas':'Sample size'})
# Creamos una columna para identificar que se está muestreando
df_agg_cand_c_est["Sampling"]='Polling places'
df_agg_cand_c_est

,Candidate with the lowest coverage,Sample size,Sampling
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,"[100, 150, 200, 250, 300, 350, 400]",Polling places


In [5]:
# Para votos
df_boostraps_v_m_c_est=df_boostraps_votos_est[["Cobertura_m_candidato_nom", "Tamaño_muestra_votos"]].copy()
df_boostraps_v_m_c_est["Tamaño_muestra_votos"]=df_boostraps_v_m_c_est["Tamaño_muestra_votos"].astype(int)
df_agg_cand_v_est=df_boostraps_v_m_c_est.groupby("Cobertura_m_candidato_nom").agg({'Tamaño_muestra_votos':'unique'}).reset_index()
# Renombramos columnas
df_agg_cand_v_est=df_agg_cand_v_est.rename(columns={'Cobertura_m_candidato_nom':'Candidate with the lowest coverage', 'Tamaño_muestra_votos':'Sample size'})
# Creamos una columna para identificar que se está muestreando
df_agg_cand_v_est["Sampling"]='Votes'
df_agg_cand_v_est

,Candidate with the lowest coverage,Sample size,Sampling
0,VIDA_ARAVARI_GOMEZ_HERRERA,[10000],Votes
1,VOTOS_NULOS_CAND_NO_REGIS,"[1000, 5000, 20000]",Votes
2,YAMIL_JASMIN_LOPEZ_MANRIQUE,"[500, 2000]",Votes


In [9]:
# Concatenamos las tablas
df_agg_com_est=pd.concat([df_agg_cand_c_est,df_agg_cand_v_est])
# Reordenamos las columnas
df_agg_com_f_est=df_agg_com_est[["Candidate with the lowest coverage","Sampling","Sample size"]].reset_index(drop=True)
df_agg_com_f_est

,Candidate with the lowest coverage,Sampling,Sample size
0,YAMIL_JASMIN_LOPEZ_MANRIQUE,Polling places,"[100, 150, 200, 250, 300, 350, 400]"
1,VIDA_ARAVARI_GOMEZ_HERRERA,Votes,[10000]
2,VOTOS_NULOS_CAND_NO_REGIS,Votes,"[1000, 5000, 20000]"
3,YAMIL_JASMIN_LOPEZ_MANRIQUE,Votes,"[500, 2000]"


In [10]:
# Renombramos los nombres de los candidatos
df_agg_com_f_est["Candidate with the lowest coverage"]=["Jasmín López Manrique", "Vida Gómez Herrera", "Null votes and votes for unregistered candidates", "Jasmín López Manrique"]
df_agg_com_f_est

,Candidate with the lowest coverage,Sampling,Sample size
0,Jasmín López Manrique,Polling places,"[100, 150, 200, 250, 300, 350, 400]"
1,Vida Gómez Herrera,Votes,[10000]
2,Null votes and votes for unregistered candidates,Votes,"[1000, 5000, 20000]"
3,Jasmín López Manrique,Votes,"[500, 2000]"


In [12]:
print(df_agg_com_f_est.to_latex(index=False, label='tab:Max_diff_cov_votos_vs_cas',caption='Maximum difference in coverages'))

\begin{table}
\caption{Maximum difference in coverages}
\label{tab:Max_diff_cov_votos_vs_cas}
\begin{tabular}{lll}
\toprule
Candidate with the lowest coverage & Sampling & Sample size \\
\midrule
Jasmín López Manrique & Polling places & [100 150 200 250 300 350 400] \\
Vida Gómez Herrera & Votes & [10000] \\
Null votes and votes for unregistered candidates & Votes & [ 1000  5000 20000] \\
Jasmín López Manrique & Votes & [ 500 2000] \\
\bottomrule
\end{tabular}
\end{table}



## Diferencia en cobertura máxima y mínima por candidato

In [13]:
# Base para la casillas inicial
df_aux_c_est=df_boostraps_c_est[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff_est = df_aux_c_est.max() - df_aux_c_est.min()

# Lo pasamos a Data Frame
df_max_c_est=pd.DataFrame(max_diff_est).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (polling places)'})
df_max_c_est

,Candidates,Maximum difference in coverages (polling places)
0,Cob_JOAQUIN_DIAZ_MENA,0.081
1,Cob_RENAN_BARRERA_CONCHA,0.077
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.109
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.094
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.200


In [14]:
# Base para la votos inicial
df_aux_v_est=df_boostraps_votos_est[["Cob_JOAQUIN_DIAZ_MENA","Cob_RENAN_BARRERA_CONCHA","Cob_VIDA_ARAVARI_GOMEZ_HERRERA","Cob_VOTOS_NULOS_CAND_NO_REGIS","Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE"]]
# Máxima diferencia por columna
max_diff_v_est = df_aux_v_est.max() - df_aux_v_est.min()

# Lo pasamos a Data Frame
df_max_v_est=pd.DataFrame(max_diff_v_est).reset_index().rename(columns={'index':'Candidates',0:'Maximum difference in coverages (votes)'})
df_max_v_est

,Candidates,Maximum difference in coverages (votes)
0,Cob_JOAQUIN_DIAZ_MENA,0.027
1,Cob_RENAN_BARRERA_CONCHA,0.026
2,Cob_VIDA_ARAVARI_GOMEZ_HERRERA,0.012
3,Cob_VOTOS_NULOS_CAND_NO_REGIS,0.045
4,Cob_YAMIL_JASMIN_LOPEZ_MANRIQUE,0.046


In [15]:
# Juntamos las bases
df_max_f_est=df_max_c_est.merge(df_max_v_est, how='inner', on='Candidates')
# Renombramos la columna de candidatos
df_max_f_est["Candidates"]=["Joaquín Díaz Mena", "Renan Barrera Concha", "Vida Gómez Herrera", "Null votes and votes for unregistered candidates","Jasmín López Manrique"]
df_max_f_est

,Candidates,Maximum difference in coverages (polling places),Maximum difference in coverages (votes)
0,Joaquín Díaz Mena,0.081,0.027
1,Renan Barrera Concha,0.077,0.026
2,Vida Gómez Herrera,0.109,0.012
3,Null votes and votes for unregistered candidates,0.094,0.045
4,Jasmín López Manrique,0.200,0.046


In [16]:
print(df_max_f_est.to_latex(index=False, label='tab:Max_diff_cov_votos_vs_cas',caption='Maximum difference in coverages'))

\begin{table}
\caption{Maximum difference in coverages}
\label{tab:Max_diff_cov_votos_vs_cas}
\begin{tabular}{lrr}
\toprule
Candidates & Maximum difference in coverages (polling places) & Maximum difference in coverages (votes) \\
\midrule
Joaquín Díaz Mena & 0.081000 & 0.027000 \\
Renan Barrera Concha & 0.077000 & 0.026000 \\
Vida Gómez Herrera & 0.109000 & 0.012000 \\
Null votes and votes for unregistered candidates & 0.094000 & 0.045000 \\
Jasmín López Manrique & 0.200000 & 0.046000 \\
\bottomrule
\end{tabular}
\end{table}

